## Steps for generating object_detect.dlc

For this demo, a YoloNAS model is used. You can read more about this model in VisionSolution1-YoloNasSSD Readme.

**Installing Necessary Libraries**

In [1]:
import os
path = "/media/code/qnn/qidk/Solutions/VisionSolution4-PoseEstimation/Generate_models"
os.chdir(path)
print(os.getcwd())

/media/code/qnn/qidk/Solutions/VisionSolution4-PoseEstimation/Generate_models


In [2]:
!pip3 install super-gradients==3.1.2
# !pip3 install cython
# !pip3 install yacs

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://mirrors.aliyun.com/pypi/simple


### Getting the dataset

In [3]:
# !wget https://github.com/ultralytics/yolov5/releases/download/v1.0/coco2017labels.zip -q --show-progress
# !wget http://images.cocodataset.org/zips/val2017.zip -q --show-progress
# !unzip val2017.zip
# !unzip coco2017labels.zip


In [4]:
!ls

GenerateDLC.ipynb	     hrnet.patch    val2017
HRNet-Human-Pose-Estimation  mode_binaries


In [5]:
import os
files = os.listdir('val2017')
for file in files[50:]:
    os.remove("val2017/"+file)

In [6]:
# %%bash
# rm -rf coco
# rm -rf coco2017labels.zip
# rm -rf val2017.zip

#### Downloading the YOLO_Nas Model

In [7]:
## Downloading Model from git repo
import torch
# Load model with pretrained weights
from super_gradients.training import models
from super_gradients.common.object_names import Models

model = models.get(Models.YOLO_NAS_S, pretrained_weights="coco")

# Prepare model for conversion
# Input size is in format of [Batch x Channels x Width x Height] where 640 is the standard COCO dataset dimensions
model.eval()
model.prep_model_for_conversion(input_size=[1, 3, 320, 320])

# Create dummy_input
dummy_input = torch.randn([1, 3, 320, 320], device="cpu")

# Convert model to onnx
torch.onnx.export(model, dummy_input, "yolo_nas_s.onnx", opset_version=11)

[2025-03-01 07:35:25] INFO - crash_tips_setup.py - Crash tips is enabled. You can set your environment variable to CRASH_HANDLER=FALSE to disable it


The console stream is logged into /home/liuqi/sg_logs/console.log


/home/liuqi/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
[2025-03-01 07:35:29] WARNING - __init__.py - Failed to import pytorch_quantization
[2025-03-01 07:35:29] WARNING - calibrator.py - Failed to import pytorch_quantization
[2025-03-01 07:35:29] WARNING - export.py - Failed to import pytorch_quantization
[2025-03-01 07:35:29] WARNING - selective_quantization_utils.py - Failed to import pytorch_quantization
[2025-03-01 07:35:29] INFO - checkpoint_utils.py - License Notification: YOLO-NAS pre-trained weights are subjected to the specific license terms and conditions detailed in 
https://github.com/Deci-AI/super-gradients/blob/master/LICENSE.YOLONAS.md
By downloading the pre

#### Converting to DLC

In [8]:
import os
os.environ['SNPE_ROOT']="/opt/qcom/aistack/qairt/2.25.0.240728"

In [9]:
%%bash
source $SNPE_ROOT/bin/envsetup.sh
snpe-onnx-to-dlc -i yolo_nas_s.onnx -o app/src/main/assets/yolo_nas_s.dlc

[INFO] AISW SDK environment set
[INFO] QNN_SDK_ROOT: /media/code/opt/qcom/aistack/qairt/2.25.0.240728
[INFO] SNPE_ROOT: /media/code/opt/qcom/aistack/qairt/2.25.0.240728


2025-03-01 07:35:40,280 - 235 - INFO - Simplified model validation is successful
2025-03-01 07:35:43,868 - 235 - INFO - INFO_INITIALIZATION_SUCCESS: 
2025-03-01 07:35:44,112 - 235 - INFO - INFO_CONVERSION_SUCCESS: Conversion completed successfully
2025-03-01 07:35:44,291 - 235 - INFO - INFO_WRITE_SUCCESS: 


## Quantizing Yolo_nas

In [10]:
##STEPS to preprocess images

def preprocess(original_image):
    resized_image = cv2.resize(original_image, (320, 320))
    resized_image = resized_image/255
    return resized_image

import cv2
import numpy as np
import os


dataset_path = "val2017/"

os.makedirs('rawYoloNAS', exist_ok=True)

filenames=[]
for path in os.listdir(dataset_path)[:5]:
    # check if current path is a file
    if os.path.isfile(os.path.join(dataset_path, path)):
        filenames.append(os.path.join(dataset_path, path))

for filename in filenames:
    original_image = cv2.imread(filename)
    img = preprocess(original_image)
    img = img.astype(np.float32)
    img.tofile("rawYoloNAS/"+filename.split("/")[-1].split(".")[0]+".raw")

In [11]:
%%bash
find rawYoloNAS -name *.raw > YoloInputlist.txt
cat YoloInputlist.txt

In [12]:
%%bash
source $SNPE_ROOT/bin/envsetup.sh
snpe-dlc-quantize --input_dlc app/src/main/assets/yolo_nas_s.dlc --input_list YoloInputlist.txt --use_enhanced_quantizer --use_adjusted_weights_quantizer --axis_quant --output_dlc app/src/main/assets/Quant_yoloNas_s_320_online.dlc

rawYoloNAS/000000198960.raw
rawYoloNAS/000000283113.raw
rawYoloNAS/000000443969.raw
rawYoloNAS/000000322429.raw
rawYoloNAS/000000188296.raw


[INFO] AISW SDK environment set
[INFO] QNN_SDK_ROOT: /media/code/opt/qcom/aistack/qairt/2.25.0.240728
[INFO] SNPE_ROOT: /media/code/opt/qcom/aistack/qairt/2.25.0.240728


[INFO] InitializeStderr: DebugLog initialized.
[WARNING] --axis_quant is deprecated, use --use_per_channel_quantization option.
[WARNING] --use_enhanced_quantizer option is deprecated, use --param_quantizer and --act_quantizer options.
[WARNING] --use_adjusted_weights_quantizer option is deprecated, use --param_quantizer option.
[INFO] Processed command-line arguments


IrQuantizer: Quantizer param type: adjusted will be deprecated in future releases
IrQuantizer: Quantizer type: adjusted is no longer supported. Using TF quantizer instead


[INFO] Quantized parameters
[INFO] Generated activations


snpe-dlc-graph-prepare requires device htp_soc info. 

So, depending on the device --htp_socs needs to be changed (sm8550 or sm8650)

In [13]:
%%bash
source $SNPE_ROOT/bin/envsetup.sh
snpe-dlc-graph-prepare --input_dlc app/src/main/assets/Quant_yoloNas_s_320_online.dlc --set_output_tensors 885,877 --output_dlc app/src/main/assets/Quant_yoloNas_s_320.dlc --htp_socs sm8650

[INFO] Saved quantized dlc to: app/src/main/assets/Quant_yoloNas_s_320_online.dlc
[INFO] DebugLog shutting down.


     3.4ms [  INFO ] Inferences will run in sync mode
     4.2ms [  INFO ] Initializing logging in the backend. Callback: [0x55a26411bb60], Log Level: [3]
     4.3ms [  INFO ] No BackendExtensions lib provided;initializing NetRunBackend Interface
     2.2ms [  INFO ] [QNN_CPU] CpuBackend creation start
     2.2ms [  INFO ] [QNN_CPU] CpuBackend creation end
     6.5ms [WARNING] Unable to find a device with NetRunDeviceKeyDefault in Library NetRunBackendLibKeyDefault
     6.5ms [WARNING] Profile Logger with name = defaultKey doesn't exist! Returning nullptr
     4.0ms [  INFO ] [QNN_CPU] QnnContext create start
     4.0ms [  INFO ] [QNN_CPU] QnnContext create end
     8.6ms [  INFO ] Entering QuantizeRuntimeApp flow
     8.6ms [WARNING] Profile Logger with name = defaultKey doesn't exist! Returning nullptr
     4.4ms [  INFO ] [QNN_CPU] CpuGraph creation start
     4.4ms [  INFO ] [QNN_CPU] CpuGraph creation end
     4.4ms [  INFO ] [QNN_CPU] QnnGraph create end
    81.7ms [  INFO ] [QNN

[INFO] AISW SDK environment set
[INFO] QNN_SDK_ROOT: /media/code/opt/qcom/aistack/qairt/2.25.0.240728
[INFO] SNPE_ROOT: /media/code/opt/qcom/aistack/qairt/2.25.0.240728


[INFO] InitializeStderr: DebugLog initialized.
[INFO] SNPE HTP Offline Prepare: Attempting to create cache for SM8650
[USER_INFO] Target device backend record identifier: HTP_V75_SM8650_8MB
[USER_INFO] No cache record in the DLC matches the target device (HTP_V75_SM8650_8MB). Creating a new record
[USER_INFO] Checking unsigned PD session
[INFO] Attempting to open dynamically linked lib: libHtpPrepare.so
[INFO] dlopen libHtpPrepare.so SUCCESS handle 0x555eebb85360
[INFO] Found Interface Provider (v2.18)
[USER_WARNING] QnnDsp <W> Initializing HtpProvider
[USER_WARNING] QnnDsp <W> HTP arch will be deprecated, please set SoC id instead.
[USER_WARNING] QnnDsp <W> Performance Estimates unsupported
[USER_INFO] Platform option not set
[USER_INFO] Created ctx=0x1 for Graph Id=0 backend=HTP SNPE Id=0x555eeb92b4e8
[USER_INFO] Offline Prepare VTCM size(MB) selected = 8
[USER_INFO] Offline Prepare Optimization Level passed = 2
[USER_WARNING] QnnDsp <W> Output padding param cannot be set explicitly.


## How to change the object-detect model ? 

Object detection models are highly dependant on model architecture, and the pre-processing requirements vary a lot from model to model. 
If user intends to use a different model e.g. YoloV5, following steps should be followed : 

- Ensure Qualcomm® Neural Processing SDK supports the operations in selected model
- Study the pre processing, and post processing requirements for the selected model
- Most object detection models operate in RGB space. Input camera YUV buffers need to be converted to RGB basd on model requirements 


# Info about HRNET

HRNET model is State-of-the-art model for human pose estimation. It has good accuracy for results with single person, but has lower accuracy for multiple persons. To enhance that, HRNET uses object-detect model to identify a single person in a frame and then give the data to HRNET to get pose of that person. In this solution, we use MobileNetSSD for detecting human and then give the preprocesssed data to HRNET to achieve better accuracy for pose estimation.

HRNET dlc takes 256x192x3 flattened array as input and returns output of dims 17x64x48. HRNET generates heatmap for 17 human joints and each heatmap is of size 64x48.

In [14]:
# %%bash
# rm -rf HRNet-Human-Pose-Estimation/
# git clone https://github.com/HRNet/HRNet-Human-Pose-Estimation.git
# git checkout 00d7bf72f56382165e504b10ff0dddb82dca6fd2
# cp hrnet.patch HRNet-Human-Pose-Estimation/
# cd HRNet-Human-Pose-Estimation/
# patch -p1 < ./hrnet.patch
# cd lib
# make

In [15]:
# %%bash
# mkdir -p mode_binaries
# cd mode_binaries
# wget https://github.com/quic/aimet-model-zoo/releases/download/hrnet-posenet/hrnet_posenet_FP32.pth

In [19]:
#####################################################################
# Getting onnx from pth model for hrnet requires a different setup  #
# python 3.6                                                        #
# torch 1.10.1                                                      #
# torchvision 0.11.2                                                #
#####################################################################

## 必须得用上面的这个torch版本!配合3.8python用

import numpy as np
from matplotlib import pyplot as plt
import sys
import torch
import torch.utils.data
import torchvision.transforms as transforms
# from config import cfg
import os
import os.path as osp
import urllib.request

%matplotlib inline


lib_path = osp.join(os.getcwd(), 'HRNet-Human-Pose-Estimation/lib')
sys.path.insert(0, lib_path)
if not os.path.exists("model_binaries"):
    os.makedirs("model_binaries")
##Getting .pth file
OPTIMIZED_CHECKPOINT_URL = (
    # "https://github.com/quic/aimet-model-zoo/releases/download/hrnet-posenet/hrnet_posenet_FP32.pth"
    "https://github.com/quic/aimet-model-zoo/releases/download/hrnet-posenet/"
)

if not os.path.exists(f"./model_binaries/hrnet_posenet_FP32.pth"):
    urllib.request.urlretrieve(
        f"{OPTIMIZED_CHECKPOINT_URL}/hrnet_posenet_FP32.pth",
        f"model_binaries/hrnet_posenet_FP32.pth",
    )


input_shape = (1, 3, 256, 192)
dummy_input = torch.randn(input_shape)
model = torch.load("model_binaries/hrnet_posenet_FP32.pth")
model.to('cpu')

onnx_model_name = "model_binaries/AIMET_HRNET_posnet.onnx"

opset = 11

# export.py use this instead
# torch.onnx.export(
#     model.cpu(),
#     dummy_input,
#     onnx_model_name,
#     verbose=True,
#     do_constant_folding=True,
#     export_params=True,
#     input_names=['input'],
#     output_names=['output'],
#     opset_version=opset
# )


## Steps for generating HRNET dlc for int8

In [20]:
%%bash
source $SNPE_ROOT/bin/envsetup.sh
snpe-onnx-to-dlc -i model_binaries/AIMET_HRNET_posnet.onnx -o app/src/main/assets/hrnet.dlc

[INFO] AISW SDK environment set
[INFO] QNN_SDK_ROOT: /media/code/opt/qcom/aistack/qairt/2.25.0.240728
[INFO] SNPE_ROOT: /media/code/opt/qcom/aistack/qairt/2.25.0.240728


2025-03-01 08:10:34,071 - 235 - INFO - Simplified model validation is successful
2025-03-01 08:10:40,989 - 235 - INFO - INFO_INITIALIZATION_SUCCESS: 
2025-03-01 08:10:41,592 - 235 - INFO - INFO_CONVERSION_SUCCESS: Conversion completed successfully


## Steps for Quantization

In [21]:
from PIL import Image
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
)

preproc = transforms.Compose(
        [
            transforms.ToTensor(),
            normalize,
        ]
    )

In [22]:

import cv2,os

dataset_path = "val2017/"

os.makedirs('rawHRNET', exist_ok=True)

filenames=[]
for path in os.listdir(dataset_path)[:5]:
    # check if current path is a file
    if os.path.isfile(os.path.join(dataset_path, path)):
        filenames.append(os.path.join(dataset_path, path))
print(filenames)

for filename in filenames:
    orig_img = cv2.imread(filename)
    img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img,(256,192),
                   interpolation = cv2.INTER_LINEAR)
    model_input = preproc(img).unsqueeze(0)

    model_input = model_input.cpu().detach().numpy()
    model_input = model_input.transpose(0,2,3,1)     
    fid = open("rawHRNET/"+filename.split("/")[-1].split(".")[0]+".raw", 'wb')
    model_input.tofile(fid)

2025-03-01 08:10:42,023 - 235 - INFO - INFO_WRITE_SUCCESS: 


['val2017/000000198960.jpg', 'val2017/000000443969.jpg', 'val2017/000000188296.jpg', 'val2017/000000322429.jpg', 'val2017/000000283113.jpg']


In [23]:
%%bash
source $SNPE_ROOT/bin/envsetup.sh

find rawHRNET -name *.raw > HRNET_input_list.txt
snpe-dlc-quantize --input_dlc app/src/main/assets/hrnet.dlc --input_list HRNET_input_list.txt --axis_quant --output_dlc app/src/main/assets/hrnet_axis_int8.dlc --enable_htp --htp_socs sm8650
snpe-dlc-info --input_dlc app/src/main/assets/hrnet_axis_int8.dlc > hrnet_axis_int8.txt

[INFO] AISW SDK environment set
[INFO] QNN_SDK_ROOT: /media/code/opt/qcom/aistack/qairt/2.25.0.240728
[INFO] SNPE_ROOT: /media/code/opt/qcom/aistack/qairt/2.25.0.240728


[INFO] InitializeStderr: DebugLog initialized.
[WARNING] --axis_quant is deprecated, use --use_per_channel_quantization option.
[INFO] Processed command-line arguments
[INFO] Quantized parameters
[INFO] Generated activations
[INFO] Saved quantized dlc to: app/src/main/assets/hrnet_axis_int8.dlc
[INFO] DebugLog shutting down.


     3.2ms [  INFO ] Inferences will run in sync mode
     3.9ms [  INFO ] Initializing logging in the backend. Callback: [0x55fee171eb60], Log Level: [3]
     3.9ms [  INFO ] No BackendExtensions lib provided;initializing NetRunBackend Interface
     2.3ms [  INFO ] [QNN_CPU] CpuBackend creation start
     2.3ms [  INFO ] [QNN_CPU] CpuBackend creation end
     6.2ms [WARNING] Unable to find a device with NetRunDeviceKeyDefault in Library NetRunBackendLibKeyDefault
     6.2ms [WARNING] Profile Logger with name = defaultKey doesn't exist! Returning nullptr
     4.0ms [  INFO ] [QNN_CPU] QnnContext create start
     4.0ms [  INFO ] [QNN_CPU] QnnContext create end
     8.1ms [  INFO ] Entering QuantizeRuntimeApp flow
     8.1ms [WARNING] Profile Logger with name = defaultKey doesn't exist! Returning nullptr
     4.3ms [  INFO ] [QNN_CPU] CpuGraph creation start
     4.3ms [  INFO ] [QNN_CPU] CpuGraph creation end
     4.3ms [  INFO ] [QNN_CPU] QnnGraph create end
   134.7ms [  INFO ] [QNN

[INFO] InitializeStderr: DebugLog initialized.
[INFO] SNPE HTP Offline Prepare: Attempting to create cache for SM8650
[USER_INFO] Target device backend record identifier: HTP_V75_SM8650_8MB
[USER_INFO] No cache record in the DLC matches the target device (HTP_V75_SM8650_8MB). Creating a new record
[USER_INFO] Checking unsigned PD session
[INFO] Attempting to open dynamically linked lib: libHtpPrepare.so
[INFO] dlopen libHtpPrepare.so SUCCESS handle 0x564c97812210
[INFO] Found Interface Provider (v2.18)
[USER_WARNING] QnnDsp <W> Initializing HtpProvider
[USER_WARNING] QnnDsp <W> HTP arch will be deprecated, please set SoC id instead.
[USER_WARNING] QnnDsp <W> Performance Estimates unsupported
[USER_INFO] Platform option not set
[USER_INFO] Created ctx=0x1 for Graph Id=0 backend=HTP SNPE Id=0x564c972bd418
[USER_INFO] Offline Prepare VTCM size(MB) selected = 8
[USER_INFO] Offline Prepare Optimization Level passed = 2
[USER_INFO] Backend Mgr ~Dtor called for backend HTP
[USER_INFO] Cleanin